# Практика 2. Дескрипторы и жизненный цикл

Теория: блоки 3–5.

Девять задач на вычисляемые атрибуты, перехват записи и на то, когда объект
создаётся, а когда исчезает.

**Как устроено.** Условие → ячейка → проверка. Ячейки, которые надо писать самому, помечены
комментарием в первой строке. Если что-то пошло не так — перезапусти ядро и выполни
служебную ячейку заново.

In [ ]:
# Служебное. Запусти эту ячейку один раз перед задачами.
def check(name, fn):
    """Запускает проверку и печатает результат."""
    try:
        fn()
        print("✓", name)
    except AssertionError as e:
        print("✗", name, "—", e)
    except Exception as e:
        print("✗", name, "—", type(e).__name__, e)


def eq(got, want):
    assert got == want, f"получено {got!r}, ожидалось {want!r}"


def expect(exc, fn):
    try:
        fn()
    except exc:
        return
    raise AssertionError(f"ожидалось {exc.__name__}")

## Задачи

### Задача 1. Что такое `cls` в `classmethod`

Один метод, два класса. Предскажи обе строки до запуска.

In [ ]:
# Ожидаемый вывод:
# Base Child
# True
class Base:
    @classmethod
    def name(cls):
        return cls.__name__


class Child(Base):
    pass


print(Base.name(), Child.name())
print(Child.name.__self__ is Child)

### Задача 2. Свойство против словаря экземпляра

В `__dict__` экземпляра лежит значение, но атрибут объявлен как `property`.
Что вернётся при чтении?

In [ ]:
# Ожидаемый вывод:
# property
# из словаря
class C:
    @property
    def x(self):
        return "property"


c = C()
c.__dict__["x"] = "из словаря"
print(c.x)
print(c.__dict__["x"])

### Задача 3. Когда вызывается `__init_subclass__`

В каком порядке напечатаются две строки: тело наследника или хук базового?

In [4]:
# Ожидаемый вывод:
# тело A
# создан A
class Base:
    def __init_subclass__(cls, **kwargs):
        super().__init_subclass__(**kwargs)
        print("создан", cls.__name__)


class A(Base):
    print("тело A")

тело A
создан A


### Задача 4. `__setattr__`, который вызывает сам себя

Класс ломается при первой же записи атрибута. Починить, не убирая хук: он должен
по-прежнему перехватывать запись.

In [5]:
class Config:
    def __setattr__(self, name, value):
        object.__setattr__(self, name, value)


c = Config()
c.x = 1

In [6]:
# Проверка. Запускай после того, как напишешь решение.
c = Config()
c.x = 1
check("значение записалось", lambda: eq(c.x, 1))
check("лежит в словаре экземпляра", lambda: eq(c.__dict__, {"x": 1}))

✓ значение записалось
✓ лежит в словаре экземпляра


### Задача 5. Проверка, которую можно обойти

`Account` проверяет баланс только в `withdraw`, поэтому отрицательный баланс ставится
напрямую. Переписать класс так, чтобы проверка работала при создании и при любой записи,
а `withdraw` продолжал работать.

In [2]:
class Account:
    def __init__(self, balance):
        self.balance = balance

    def __setattr__(self, name, value):
        if name == "balance" and value < 0:
            raise ValueError("баланс не может быть отрицательным")
        object.__setattr__(self, name, value)

    def withdraw(self, amount):
        if self.balance - amount < 0:
            raise ValueError("недостаточно средств")
        self.balance -= amount

In [3]:
# Проверка. Запускай после того, как напишешь решение.
check("отрицательный при создании", lambda: expect(ValueError, lambda: Account(-1)))
a = Account(100)
a.withdraw(40)
check("списание работает", lambda: eq(a.balance, 60))
check("отрицательный при записи", lambda: expect(ValueError, lambda: setattr(a, "balance", -5)))
check("после отказа баланс прежний", lambda: eq(a.balance, 60))

✓ отрицательный при создании
✓ списание работает
✓ отрицательный при записи
✓ после отказа баланс прежний


### Задача 6. Дескриптор с диапазоном

**Контракт.** `Bounded(low, high)` — дескриптор, который проверяет значение при записи:

- значение вне диапазона → `ValueError`;
- значение хранится **в экземпляре**, а не в дескрипторе: один дескриптор обслуживает
  сколько угодно объектов;
- `Account` объявляет `balance = Bounded(0, 1_000_000)` и `rate = Bounded(0, 100)`;
- `Account(balance=..., rate=...)` проверяет оба значения при создании.

Сигнатуры даны, тела — твои.

In [ ]:
class Bounded:
    def __init__(self, low, high):
        self.low = low
        self.high = high

    def __set_name__(self, owner, name):
        self.storage_name = f"_{name}"

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        return instance.__dict__[self.storage_name]

    def __set__(self, instance, value):
        if not self.low <= value <= self.high:
            raise ValueError("значение вне диапазона")
        instance.__dict__[self.storage_name] = value


class Account:
    balance = Bounded(0, 1_000_000)
    rate = Bounded(0, 100)

    def __init__(self, balance, rate):
        self.balance = balance
        self.rate = rate

In [ ]:
# Проверка. Запускай после того, как напишешь решение.
check("баланс при создании", lambda: expect(ValueError, lambda: Account(balance=-1, rate=10)))
check("ставка при создании", lambda: expect(ValueError, lambda: Account(balance=10, rate=101)))
a = Account(balance=100, rate=10)
check("значения читаются", lambda: eq((a.balance, a.rate), (100, 10)))
check("запись вне диапазона", lambda: expect(ValueError, lambda: setattr(a, "balance", 2_000_000)))
b = Account(balance=5, rate=1)
check("экземпляры не делят значение", lambda: eq((a.balance, b.balance), (100, 5)))

### Задача 7. Кэш экземпляров

**Контракт.** `Point(x, y)`:

- `Point(1, 2) is Point(1, 2)` → `True`;
- для разных координат — разные объекты;
- координаты доступны как `p.x` и `p.y`.

Такое кэширование делают через `__new__`: `__init__` вызывается уже после того, как объект
создан, и на повторный вызов не влияет.

In [ ]:
class Point:
    _cache = {}

    def __new__(cls, x, y):
        key = (cls, x, y)
        if key not in Point._cache:
            point = super().__new__(cls)
            point.x, point.y = x, y
            Point._cache[key] = point
        return Point._cache[key]

In [ ]:
# Проверка. Запускай после того, как напишешь решение.
p1 = Point(1, 2)
p2 = Point(1, 2)
p3 = Point(2, 1)
check("одинаковые координаты — один объект", lambda: eq(p1 is p2, True))
check("разные координаты — разные объекты", lambda: eq(p1 is p3, False))
check("координаты на месте", lambda: eq((p1.x, p1.y), (1, 2)))

### Задача 8. Почему property победил словарь

В задаче 2 значение лежало в `__dict__` экземпляра, но чтение вернуло `property`.
Объясни порядок поиска атрибута и место дескрипторов в нём.

**Ответ.** При чтении Python сначала ищет в классе data descriptor и вызывает его, затем проверяет словарь экземпляра, а после — обычные атрибуты класса и non-data descriptors. `property` — data descriptor, поэтому он возвращает `"property"` раньше, чем Python обращается к `c.__dict__["x"]`.

### Задача 9. Фабрика, которая не знает о наследнике

`Child.create()` возвращает `Base`. Починить так, чтобы каждый класс создавал себя,
не дублируя метод в наследнике.

In [7]:
class Base:
    @classmethod
    def create(cls):
        return cls()


class Child(Base):
    pass

In [8]:
# Проверка. Запускай после того, как напишешь решение.
check("наследник создаёт наследника", lambda: eq(type(Child.create()).__name__, "Child"))
check("базовый создаёт базовый", lambda: eq(type(Base.create()).__name__, "Base"))

✓ наследник создаёт наследника
✓ базовый создаёт базовый
